# Verification Summary

This notebook requires all component reports, rejects failed or stale
reports, and displays skipped checks separately from required passing checks.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


Repository root: /Users/llsra/Desktop/DeepGPR
DeepGPR package: /Users/llsra/Desktop/DeepGPR/src/DeepGPR/__init__.py


In [2]:
import json

current_metadata = vu.runtime_metadata(DeepGPR, "cpu")
freshness_keys = (
    "native_abi",
    "source_tree_sha256",
    "native_library_sha256",
)

expected_reports = [
    "00_local_backend_and_contracts",
    "01_forward_physics",
    "02_cpml_absorption",
    "03_gradient_2d",
    "04_gradient_3d",
    "05_wavefield_storage",
    "06_cpu_cuda_parity",
    "07_long_run_stability",
    "08_openmp_parallelism",
    "09_anisotropic_grid",
]
reports = []
for report_name in expected_reports:
    path = vu.RESULTS_DIR / f"{report_name}.json"
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing report {path}. Run the component notebooks in order."
        )
    report = json.loads(path.read_text())
    if report.get("overall_status") != "PASS":
        raise AssertionError(f"Report failed: {path}")
    report_metadata = report.get("metadata", {})
    mismatches = {
        key: {
            "report": report_metadata.get(key),
            "current": current_metadata.get(key),
        }
        for key in freshness_keys
        if report_metadata.get(key) != current_metadata.get(key)
    }
    if mismatches:
        raise RuntimeError(
            f"Stale report {path}. Re-run its component notebook. "
            f"Metadata mismatches: {mismatches}"
        )
    reports.append(report)


In [3]:
total_passed = 0
total_skipped = 0
print(f"{'Report':38s} {'Passed':>8s} {'Skipped':>8s} {'Status':>8s}")
print("-" * 68)
for report in reports:
    passed = sum(item["status"] == "PASS" for item in report["checks"])
    skipped = sum(item["status"] == "SKIPPED" for item in report["checks"])
    total_passed += passed
    total_skipped += skipped
    print(
        f"{report['report']:38s} {passed:8d} {skipped:8d} "
        f"{report['overall_status']:>8s}"
    )
print("-" * 68)
print(f"{'Total':38s} {total_passed:8d} {total_skipped:8d} {'PASS':>8s}")

cuda_report = next(
    report for report in reports if report["report"] == "06_cpu_cuda_parity"
)
cuda_skipped = any(
    item["status"] == "SKIPPED" for item in cuda_report["checks"]
)
print(f"CUDA parity evidence present: {not cuda_skipped}")


Report                                   Passed  Skipped   Status
--------------------------------------------------------------------
00_local_backend_and_contracts                9        0     PASS
01_forward_physics                            8        0     PASS
02_cpml_absorption                            6        0     PASS
03_gradient_2d                                9        0     PASS
04_gradient_3d                               28        0     PASS
05_wavefield_storage                          7        1     PASS
06_cpu_cuda_parity                            1        1     PASS
07_long_run_stability                         6        0     PASS
08_openmp_parallelism                         5        0     PASS
09_anisotropic_grid                          15        1     PASS
--------------------------------------------------------------------
Total                                        94        3     PASS
CUDA parity evidence present: False


In [4]:
summary_metadata = {
    "component_reports": expected_reports,
    "passed_checks": total_passed,
    "skipped_checks": total_skipped,
    "cuda_parity_evidence_present": not cuda_skipped,
}
vu.save_report(
    "99_verification_summary",
    [
        {
            "name": "all component reports passed",
            "status": "PASS",
            **summary_metadata,
        }
    ],
    {**current_metadata, "repository_root": str(REPO_ROOT)},
)


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/99_verification_summary.json


PosixPath('/Users/llsra/Desktop/DeepGPR/tests/results/99_verification_summary.json')